# MST Corpus 10m Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 20.06.2026 (by Sebastian Birk; sb75@sanger.ac.uk)

Run this notebook in the terra-rapids UV environment, installable from ```/nfs/team361/sb75/terra-reproducibility/envs/terra-rapids.sh```:

```source /nfs/team361/sb75/.venvs/terra-rapids/bin/activate```

# 1. Setup

## 1.1 Load Libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../utils")

In [3]:
import logging
import math
import os
import pickle
import random
import re
import warnings
from collections import defaultdict
from datetime import datetime
from itertools import chain, compress
from pathlib import Path

import anndata as ad
import celltypist
import dask
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rapids_singlecell as rsc
import scanpy as sc
import scib_metrics
import seaborn as sns
import torch

dask.config.set({"dataframe.query-planning": True})
import dask.dataframe.dask_expr
import squidpy as sq
import rapids_singlecell as rsc

from datasets import concatenate_datasets, load_from_disk
from scib_metrics import ilisi_knn
from scib_metrics.nearest_neighbors import pynndescent
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from app.inference import get_gene_embed, get_average_gene_embed, infer, embed_dataset
from app.training import train
from terra.datasets import prepare_dataset
from terra.utils.config import create_params_from_YAML_wandb_config
from terra.utils.distributed import init_distributed
from terra.utils.embedding import collect_adata_from_folder
from terra.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

/nfs/team361/sb75/.venvs/terra-rapids/lib/python3.12/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv
/nfs/team361/sb75/.venvs/terra-rapids/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
JAX version 0.10.1 available.


## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [4]:
### Set params
dataset = 'mst_corpus_10m'
tokenizer = 'cell_graph'
#load_timestamp = '09062026_123006_288'
#read_checkpoint = 'terra-ep0.pth.tar' # x, Weighted NMI is 0.5725112949736703. Weighted ARI is 0.2089251173363624.
#read_checkpoint = 'terra-ep1.pth.tar' # 
#read_checkpoint = 'terra-ep2.pth.tar' # Weighted NMI is 0.626296212960561. Weighted ARI is 0.30606221985927134; Weighted NMI is 0.6327897712396886. Weighted ARI is 0.28584177555885887.
#read_checkpoint = 'terra-ep3.pth.tar' # Weighted NMI is 0.6514086064776036. Weighted ARI is 0.32671788619449144. 
#read_checkpoint = 'terra-ep6.pth.tar' # Weighted NMI is 0.6385495279508457. Weighted ARI is 0.30460549611026305; Weighted NMI is 0.6302880632815105. Weighted ARI is 0.31025550324915874.
#read_checkpoint = 'terra-ep7.pth.tar' # Weighted NMI is 0.6380189309468287. Weighted ARI is 0.3085541099922999; Weighted NMI is 0.6421661932084666. Weighted ARI is 0.3089642703186703.
#read_checkpoint = 'terra-ep8.pth.tar' # Weighted NMI is 0.6453198394695641. Weighted ARI is 0.30381481660263937; Weighted NMI is 0.643566673401051. Weighted ARI is 0.3088747939221945.
#read_checkpoint = 'terra-ep9.pth.tar' # Weighted NMI is 0.6521588930706833. Weighted ARI is 0.3172710552446482.

#load_timestamp = '17062026_224730_174'
#read_checkpoint = 'terra-ep0.pth.tar' # Weighted NMI is 0.6136385108207226. Weighted ARI is 0.267537499595329; Weighted NMI is 0.6182555764373384. Weighted ARI is 0.26377778024418597.
#read_checkpoint = 'terra-ep4.pth.tar' # Weighted NMI is 0.588640479016241. Weighted ARI is 0.2639385999669161; Weighted NMI is 0.5940487224147768. Weighted ARI is 0.24250655814671812.
#read_checkpoint = 'terra-ep9.pth.tar' # Weighted NMI is 0.5743655236065558. Weighted ARI is 0.2599392030760015; Weighted NMI is 0.5832059542701941. Weighted ARI is 0.20634314797219735; 
#read_checkpoint = 'terra-ep14.pth.tar' # Weighted NMI is 0.6348632484958024. Weighted ARI is 0.2994703450216038; Weighted NMI is 0.6343544330385399. Weighted ARI is 0.29728326973504193.

load_timestamp = '21042026_092950_827'
read_checkpoint = 'terra-ep2.pth.tar'

emb_layer = None
top_k = None
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
batch_size = 256

In [5]:
# Define paths
artifacts_folder_path = '/nfs/team361/sb75/terra-reproducibility/artifacts'
datasets_folder_path = '/nfs/team361/sb75/DATASETS'
raw_data_folder_path = f'/nfs/team361/sb75/DATASETS/silver/{dataset}'

# Define folder path for saving and loading artifacts
model_folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'

In [6]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [7]:
# Load config
params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml', logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
    params['state']['load_checkpoint'] = True

layer = params['meta']['enc_depth']
params['data']['batch_size'] = batch_size

print(params)

Loaded parameters from YAML file.


{'data': {'batch_size': 256, 'count_cell_norm_method': None, 'count_count_norm_method': None, 'count_gene_norm_method': None, 'dataset_name': 'mst_corpus_10m', 'mega_batch_mult_max': 1000, 'model_input_size': 2816, 'n_neighs': 10, 'n_segments': 11, 'n_special_values': 1000, 'num_workers': 16, 'nz_spc': True, 'pin_memory': True, 'precomputed_n_nonzero_tokens': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/mst_corpus_10m_32_None_None_None_None_None_None_knn_10_2816_n_nonzero_tokens_train.pkl', 'precomputed_split': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/mst_corpus_10m_32_None_None_None_None_None_None_knn_10_2816_precomputed_split_train.pkl', 'random_state': 0, 'rank_cell_norm_method': None, 'rank_count_norm_method': None, 'rank_gene_norm_method': None, 'raw_data_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mst_corpus_10m', 'sample_subset': False, 'sampling_strategy': None, 'sep_gene_tokens_neb': False, 'seq_len_cell': 256, 'seq_len_ne

## 1.3 Run Notebook Setup

In [8]:
os.makedirs(f'{model_folder_path}/figures', exist_ok=True)

In [9]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [10]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [11]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [12]:
sc.settings.figdir = f'{model_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

## 1.4 Define Functions

In [13]:
def make_large_cmap(n):
    base = mpl.colormaps["turbo"].resampled(n)   # replaces plt.cm.get_cmap("turbo", n)
    return [mcolors.to_hex(base(i)) for i in range(n)]

In [14]:
def run_inference(dataset_name,
                  params,
                  read_checkpoint,
                  obs_cols,
                  emb_layers,
                  #n_included_cells_list,
                  top_k,
                  rerun_inference=False,
                  write_to_disk=True):
    # Load inference results if they exist
    if os.path.exists(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad') and not rerun_inference:
        adata = sc.read_h5ad(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    else:
        # Load data
        if params['data']['precomputed_split']:
            test_dataset, _, _ = prepare_dataset(params)
        else:
            _, _, test_dataset = prepare_dataset(params)
        test_dataset
        
        # Run model inference
        adata = infer(params,
                      test_dataset,
                      load_folder_path=model_folder_path,
                      dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
                      obs_cols=obs_cols,
                      emb_layers=emb_layers,
                      agg_type='avg',
                      #n_included_cells_list=n_included_cells_list,
                      top_k=top_k,
                      masked_tokens=None,
                      agg_excluded_genes=None,
                      include_spatial_cell_emb=True,
                      cell_gene_ids=[],
                      neighborhood_gene_ids=[],
                      ignore_spc_tokens=True)

    # Extract batch from cell ID
    adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]
    
    # Remove missing ground truth
    #for obs_col in obs_cols:
    #    adata = adata[~adata.obs[obs_col].isna()]

    if write_to_disk:
        # Save adata
        adata.write(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [15]:
def compute_embedding_similarity(adata,
                                 emb_layer,
                                 emb_keys=['cell', 'neighborhood'],
                                 label='niche_type',
                                 write_to_disk=True,
                                 ):
    for emb_key in emb_keys:
        similarity_stats = compute_similarity_stats(
            adata,
            n_samples=None,
            type_key=label,
            embedding_key=f'{emb_key}_emb_layer_{emb_layer}')

        avg_cos_sim = np.mean(similarity_stats["Mean Similarity"])
    
        cosine_df = pd.DataFrame({
            'metric': [f'avg_{label}_cos_sim'],
            'value': [avg_cos_sim]
        })
        print(f'Average {label} cosine similarity is {avg_cos_sim}.')
        cosine_df.to_csv(f'{model_folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_avg_{label}_cos_sim.csv',
                         index=False)
        
        plot_similarity_heatmaps(
            similarity_stats,
            save_file_path=f'{model_folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_emb_{label}_cos_sim.svg')

In [16]:
def compute_umap(adata,
                 emb_keys,
                 emb_layer,
                 write_to_disk=True):
    for emb_key in emb_keys:
        if f'{emb_key}_connectivities' not in adata.obsp.keys():
            # Compute UMAP representation for aggregated gene embeddings
            rsc.pp.neighbors(adata,
                             n_neighbors=15,
                             use_rep=f'{emb_key}_emb_layer_{emb_layer}',
                             key_added=emb_key)
        if f'{emb_key}_umap' not in adata.obsm.keys():
            rsc.tl.umap(adata,
                        neighbors_key=emb_key,
                        key_added=f'{emb_key}_umap')
        
    if write_to_disk:
        # Save adata
        adata.write(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
                
    return adata

In [17]:
def compute_latent_clusters(adata,
                            emb_keys=['cell', 'neighborhood'],
                            obs_col='niche_type',
                            n_unique_vals=None,
                            latent_leiden_resolution_cell=1.0,
                            latent_leiden_resolution_neighborhood=0.4,
                            latent_leiden_resolution_agg=0.4,
                            write_to_disk=True):
    for emb_key in emb_keys:
        latent_leiden_resolutions = []
        if emb_key == 'cell' or 'ctx_1' in emb_key:
            latent_leiden_resolution = latent_leiden_resolution_cell
        elif emb_key == 'neighborhood' or 'ctx' in emb_key:
            latent_leiden_resolution = latent_leiden_resolution_neighborhood
        elif emb_key == 'agg':
            latent_leiden_resolution = latent_leiden_resolution_agg        
        adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']
        # Perform clustering until number of ground truth niches is matched
        i = 0
        if not n_unique_vals:
            n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
        print(f'Target number of unique values: {n_unique_vals}.')
        n_latent_clusters = 0
        while (n_latent_clusters != n_unique_vals):
            if latent_leiden_resolution in latent_leiden_resolutions:
                latent_leiden_resolution += random.uniform(0.0001, 0.001)
            latent_leiden_resolutions.append(latent_leiden_resolution)
            print(f'Leiden resolution: {latent_leiden_resolution}.')
            latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
            i += 1
            rsc.tl.leiden(adata,
                          neighbors_key=emb_key,
                          key_added=latent_cluster_key,
                          resolution=latent_leiden_resolution)
        
            n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
            print(f"Number of latent clusters {n_latent_clusters}.")
            if n_latent_clusters == n_unique_vals:
                break
            elif n_latent_clusters < (n_unique_vals - 8):
                latent_leiden_resolution += 0.1
            elif n_latent_clusters < (n_unique_vals - 5):
                latent_leiden_resolution += 0.05
            elif n_latent_clusters < (n_unique_vals - 3):
                latent_leiden_resolution += 0.02
            elif n_latent_clusters < n_unique_vals:
                latent_leiden_resolution += 0.001
            elif n_latent_clusters > (n_unique_vals + 8):
                latent_leiden_resolution -= 0.1
            elif n_latent_clusters > (n_unique_vals + 5):
                latent_leiden_resolution -= 0.05
            elif n_latent_clusters > (n_unique_vals + 3):
                latent_leiden_resolution -= 0.02
            else:
                latent_leiden_resolution -= 0.001
        
        # Compute metrics and store to disk
        metrics = []
        
        # Per-batch metrics
        for batch in ["15", "82"]:
            adata_batch = adata[adata.obs["batch"] == batch].copy()
            print(adata_batch)
        
            labels = adata_batch.obs[obs_col]
            labels_pred = adata_batch.obs[latent_cluster_key]
        
            nmi = normalized_mutual_info_score(
                labels,
                labels_pred,
                average_method="arithmetic"
            )
            ari = adjusted_rand_score(labels, labels_pred)
        
            n_cells = adata_batch.n_obs
        
            print(f"Batch {batch}: NMI is {nmi}. ARI is {ari}.")
        
            metrics.append({
                "subset": f"batch_{batch}",
                "batch": batch,
                "n_cells": n_cells,
                "metric": "NMI",
                "value": nmi,
            })
        
            metrics.append({
                "subset": f"batch_{batch}",
                "batch": batch,
                "n_cells": n_cells,
                "metric": "ARI",
                "value": ari,
            })
    
        # Weighted averages across batches
        batch_metrics_df = pd.DataFrame(metrics)
        
        weighted_nmi = (
            batch_metrics_df[
                (batch_metrics_df["batch"] != "all") &
                (batch_metrics_df["metric"] == "NMI")
            ]
            .assign(weighted_value=lambda df: df["value"] * df["n_cells"])
        )
        
        weighted_nmi = weighted_nmi["weighted_value"].sum() / weighted_nmi["n_cells"].sum()
        
        weighted_ari = (
            batch_metrics_df[
                (batch_metrics_df["batch"] != "all") &
                (batch_metrics_df["metric"] == "ARI")
            ]
            .assign(weighted_value=lambda df: df["value"] * df["n_cells"])
        )
        
        weighted_ari = weighted_ari["weighted_value"].sum() / weighted_ari["n_cells"].sum()
        
        print(f"Weighted NMI is {weighted_nmi}. Weighted ARI is {weighted_ari}.")
        
        metrics.append({
            "subset": "weighted_batches",
            "batch": "weighted",
            "n_cells": adata.n_obs,
            "metric": "weighted_NMI",
            "value": weighted_nmi,
        })
        
        metrics.append({
            "subset": "weighted_batches",
            "batch": "weighted",
            "n_cells": adata.n_obs,
            "metric": "weighted_ARI",
            "value": weighted_ari,
        })
        
        
        # Save to CSV
        metrics_df = pd.DataFrame(metrics)
        
        metrics_df.to_csv(
            f'{model_folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
            index=False
        )
        
        # Plot UMAPs
        os.makedirs(f'{model_folder_path}/figures', exist_ok=True)
        labels = ['batch', obs_col, latent_cluster_key]
        # Plot (aggregated) neighborhood gene embeddings, colored by labels
        for label in labels:
            if len(np.unique(adata.obs[label])) > 1:
                n_cats = adata.obs[label].nunique()
                palette = make_large_cmap(n_cats)
                sc.pl.umap(adata,
                           neighbors_key=emb_key,
                           title=(emb_key + "_emb_" + label if emb_key not in label else label),
                           color=label,
                           palette=(
                                custom_palette
                                if label == latent_cluster_key
                                else palette if label == obs_col
                                else None
                            ),
                           show=False)
                plt.savefig(f'{model_folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
                            dpi=300,
                            bbox_inches='tight')
        
        # Plot niches in tissue
        batch_ids = adata.obs['batch'].unique().tolist()
        for i, batch_id in enumerate(batch_ids):
            adata_batch = adata[adata.obs['batch'] == batch_id]
            sc.pl.spatial(adata_batch,
                          color=latent_cluster_key,
                          palette=custom_palette,
                          spot_size=spot_sizes[i],
                          save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')
            os.rename(f'{model_folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
                      f'{model_folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))
    
        if write_to_disk:
            # Save adata
            adata.write(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    return adata, latent_cluster_key

In [18]:
def mmd_comparable(adata, emb_key, batch_key, n_sub=2000, n_sigma=1000):
    # Normalize to zero mean, unit variance
    X = StandardScaler().fit_transform(adata.obsm[emb_key])
    
    # Compute sigma on normalized data
    idx = np.random.choice(len(X), min(n_sigma, len(X)), replace=False)
    sub = X[idx]
    dists = np.linalg.norm(sub[:, None] - sub[None, :], axis=2)
    sigma = np.median(dists[dists > 0])
    
    # Split by batch
    batches = adata.obs[batch_key].unique()
    mask0 = adata.obs[batch_key] == batches[0]
    mask1 = adata.obs[batch_key] == batches[1]
    b0, b1 = X[mask0], X[mask1]
    
    idx0 = np.random.choice(len(b0), min(n_sub, len(b0)), replace=False)
    idx1 = np.random.choice(len(b1), min(n_sub, len(b1)), replace=False)
    
    def rbf(A, B):
        d = np.linalg.norm(A[:, None] - B[None, :], axis=2)
        return np.exp(-d**2 / (2 * sigma**2))
    
    b0s, b1s = b0[idx0], b1[idx1]
    return rbf(b0s, b0s).mean() + rbf(b1s, b1s).mean() - 2 * rbf(b0s, b1s).mean()

In [19]:
def compute_batch_integration_metrics(
    adata,
    emb_key,
    batch_key,
    dataset_name,
    model_folder_path,
    n_neighbors=90,
    sample_size=10000,
):
    """
    Compute batch integration metrics and save them to CSV.

    Metrics:
    - iLISI
    - ASW batch
    - MMD
    """
    results = []

    # iLISI
    batch = adata.obs[batch_key].values
    knn = pynndescent(adata.obsm[emb_key], n_neighbors=n_neighbors)
    ilisi = float(np.mean(ilisi_knn(knn, batch)))
    print(f"iLISI: {ilisi:.4f}")
    results.append({"metric": "iLISI", "score": ilisi})

    # ASW
    asw = silhouette_score(
        adata.obsm[emb_key],
        adata.obs[batch_key],
        metric="euclidean",
        sample_size=sample_size,
    )
    print(f"ASW batch: {asw:.4f}")
    results.append({"metric": "ASW", "score": asw})

    # MMD
    mmd_score = mmd_comparable(adata, emb_key, batch_key)
    print(f"MMD: {mmd_score:.4f}")
    results.append({"metric": "MMD", "score": mmd_score})

    # Save
    df = pd.DataFrame(results)
    output_path = f"{model_folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_batch_integration_metrics.csv"
    df.to_csv(output_path, index=False)

    return df

# 2. Analysis

### 2.1 mmb0-1b_smb1-1b (MERFISH & STARmap Mouse Brain)

In [20]:
# Set dataset params
dataset_name = 'mst_corpus_10m' # 'mst_corpus_10m', 'mmb0-1b_smb1-1b_1p', 'mmb0-1b_smb1-1b_2p'
obs_cols = ['ccf_region_name', 'Sub_molecular_tissue_region']
niche_key = 'niche'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [100, 20]

#### 2.1.1 Define Config

In [21]:
########## Change inference data ##########
if dataset_name == 'mmb0-1b_smb1-1b_1p':
    params['data']['raw_data_folder_path'] = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mmb0-1b_smb1-1b_1p'
    params['data']['tokenized_data_folder_path'] = params['data']['tokenized_data_folder_path'].replace('mst_corpus_10m', 'mmb0-1b_smb1-1b_1p')
elif dataset_name == 'mmb0-1b_smb1-1b_2p':
    params['data']['raw_data_folder_path'] = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mmb0-1b_smb1-1b_2p'
    params['data']['tokenized_data_folder_path'] = params['data']['tokenized_data_folder_path'].replace('mst_corpus_10m', 'mmb0-1b_smb1-1b_2p')
else:
    pass
###########################################

# Set test batch IDs
params['data']['precomputed_split'] = None
params['data']['precomputed_n_nonzero_tokens'] = None
params['data']['test_batch_ids'] = [
    '0_batch82',
    '1_batch15']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

print(params['data'])

{'batch_size': 256, 'count_cell_norm_method': None, 'count_count_norm_method': None, 'count_gene_norm_method': None, 'dataset_name': 'mst_corpus_10m', 'mega_batch_mult_max': 1000, 'model_input_size': 2816, 'n_neighs': 10, 'n_segments': 11, 'n_special_values': 1000, 'num_workers': 16, 'nz_spc': True, 'pin_memory': True, 'precomputed_n_nonzero_tokens': None, 'precomputed_split': None, 'random_state': 0, 'rank_cell_norm_method': None, 'rank_count_norm_method': None, 'rank_gene_norm_method': None, 'raw_data_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mst_corpus_10m', 'sample_subset': False, 'sampling_strategy': None, 'sep_gene_tokens_neb': False, 'seq_len_cell': 256, 'seq_len_neighborhood': 2560, 'split': 0, 'stratify': False, 'test_batch_ids': ['0_batch82', '1_batch15'], 'token_dict_folder_path': '/nfs/team361/sb75/DATASETS/tokenizer/token_dictionary_mus_musculus.pkl', 'tokenized_data_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-to

#### 2.1.2 Run Model Inference

In [22]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=obs_cols,
                      emb_layers=[params['meta']['enc_depth']], # list(range(1, params['meta']['enc_depth'] + 1)),
                      top_k=top_k,
                      write_to_disk=True)

Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.


None


EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(25263, 384, padding_idx=0)
    (seg_embed): Embedding(13, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
 

Special token pad ratio: 1.0


Dataloader and -sampler created.
Loaded pretrained target encoder from epoch 3 with msg: <All keys matched successfully>.
Finished loading checkpoint with read path: /nfs/team361/sb75/terra-reproducibility/artifacts/mst_corpus_10m/21042026_092950_827/terra-ep2.pth.tar.


['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr']


1it [00:01,  1.02s/it]


Loading metadata AnnDatas...
Loading AnnData objects from /lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mst_corpus_10m.


... storing 'batch' as categorical


Layer 12 cosine:
tensor([0.5271, 0.8166, 0.9035])


In [23]:
adata.obs[niche_key] = adata.obs["ccf_region_name"].combine_first(
    adata.obs["Sub_molecular_tissue_region"]
)            

# Filter cells without niche labels
adata = adata[~adata.obs[niche_key].isna()]

#### 2.1.3 Identify Niches

In [24]:
adata = compute_umap(adata,
                     emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
                     emb_layer=params['meta']['enc_depth'], # 'agg',
                     write_to_disk=True)

ValueError: n_neighbors must be <= number of samples in index

In [ ]:
adata, latent_cluster_key = compute_latent_clusters(
    adata,
    emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
    obs_col=niche_key,
    n_unique_vals=30,
    latent_leiden_resolution_cell=latent_leiden_resolution_cell,
    latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
    write_to_disk=True)

#### 2.1.4 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'], # ['neighborhood'], ['agg']
                             emb_layer=params['meta']['enc_depth'], # 'agg',
                             label=latent_cluster_key,
                             write_to_disk=True)

In [ ]:
Z = adata.obsm[f'cell_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

In [ ]:
Z = adata.obsm[f'neighborhood_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

#### 2.1.5 Compute Data Integration Metrics

In [ ]:
batch_metrics_df = compute_batch_integration_metrics(
    adata=adata,
    emb_key=f'neighborhood_emb_layer_{params['meta']['enc_depth']}',
    batch_key=batch_key,
    dataset_name=dataset_name,
    model_folder_path=model_folder_path,
)